In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import os
os.chdir("/workspaces/TFG_Spain-s_Migratory_Flow")

RAW    = Path("output/evr/01-raw")
SILVER = Path("output/evr/02-silver")
SILVER.mkdir(parents=True, exist_ok=True)
print("✅ Rutas OK")

✅ Rutas OK


In [2]:
#============================================================
# Celda 2 — Salidas (total por provincia origen)
#============================================================
df = pd.read_parquet(RAW / "raw_flujo_interprovincial.parquet")

EXCLUIR = {"Total Nacional", "Flujo de migraciones interprovinciales"}

salidas = (
    df[
        (df['sexo'] == 'Total') &
        (df['provincia_destino'] == 'Total Nacional') &
        (~df['provincia_origen'].isin(EXCLUIR))
    ]
    [['provincia_origen', 'anyo', 'valor']]
    .rename(columns={'provincia_origen': 'provincia', 'valor': 'salidas'})
    .reset_index(drop=True)
)

print(f"Salidas shape: {salidas.shape}  → esperado 728 (52×14)")
print(f"Provincias: {salidas['provincia'].nunique()}")
print(f"\nMadrid:")
print(salidas[salidas['provincia']=='Madrid'].sort_values('anyo').to_string(index=False))

Salidas shape: (728, 3)  → esperado 728 (52×14)
Provincias: 52

Madrid:
provincia  anyo  salidas
   Madrid  2008  79288.0
   Madrid  2009  73490.0
   Madrid  2010  72549.0
   Madrid  2011  67771.0
   Madrid  2012  62397.0
   Madrid  2013  59207.0
   Madrid  2014  57410.0
   Madrid  2015  55077.0
   Madrid  2016  55188.0
   Madrid  2017  54993.0
   Madrid  2018  62471.0
   Madrid  2019  67176.0
   Madrid  2020  70288.0
   Madrid  2021  78600.0


In [3]:
#============================================================
# Celda 3 — Entradas (transponer la matriz OD via cumcount)
#============================================================

# Lista de provincias en el MISMO orden que el INE las emite como salidas
# (orden alfabético del raw, que es el orden de filas en la tabla OD)
provincias_ordenadas = sorted(
    df[
        (df['sexo'] == 'Total') &
        (df['provincia_destino'] == 'Total Nacional') &
        (~df['provincia_origen'].isin(EXCLUIR))
    ]['provincia_origen'].unique().tolist()
)

# Para cada año, las filas con origen=X y destino="Flujo..."
# representan cuánto llega a cada provincia destino desde X.
# Si sumamos por destino: entradas_destino_Y = sum(flujo[X→Y]) para todo X≠Y
# En este raw el destino real NO está explícito, pero:
# Las filas de origen="Total Nacional", destino="Flujo..." son exactamente
# la suma de columnas: cuánto llega a cada provincia desde todo el país.
# Esas 52 filas por año están en el mismo orden que provincias_ordenadas.

entradas_raw = (
    df[
        (df['sexo'] == 'Total') &
        (df['provincia_origen'] == 'Total Nacional') &
        (df['provincia_destino'] == 'Flujo de migraciones interprovinciales')
    ]
    [['anyo', 'valor']]
    .sort_values(['anyo'])
    .reset_index(drop=True)
    .copy()
)

# Asignar provincia por posición dentro de cada año usando cumcount
entradas_raw['pos'] = entradas_raw.groupby('anyo').cumcount()
entradas_raw['provincia'] = entradas_raw['pos'].map(
    dict(enumerate(provincias_ordenadas))
)

entradas = (
    entradas_raw[['provincia', 'anyo', 'valor']]
    .rename(columns={'valor': 'entradas'})
    .reset_index(drop=True)
)

print(f"Entradas shape: {entradas.shape}  → esperado 728 (52×14)")
print(f"Provincias: {entradas['provincia'].nunique()}")
print(f"\nVerificación — filas por año (deben ser 52):")
print(entradas.groupby('anyo').size().to_string())
print(f"\nMadrid:")
print(entradas[entradas['provincia']=='Madrid'].sort_values('anyo').to_string(index=False))

Entradas shape: (728, 3)  → esperado 728 (52×14)
Provincias: 52

Verificación — filas por año (deben ser 52):
anyo
2008    52
2009    52
2010    52
2011    52
2012    52
2013    52
2014    52
2015    52
2016    52
2017    52
2018    52
2019    52
2020    52
2021    52

Madrid:
provincia  anyo  entradas
   Madrid  2008   17936.0
   Madrid  2009   10071.0
   Madrid  2010    4125.0
   Madrid  2011    3871.0
   Madrid  2012    5822.0
   Madrid  2013    3703.0
   Madrid  2014    5705.0
   Madrid  2015   13311.0
   Madrid  2016    6379.0
   Madrid  2017    1639.0
   Madrid  2018    1771.0
   Madrid  2019   17324.0
   Madrid  2020    5017.0
   Madrid  2021    8115.0


In [4]:
#============================================================
# Celda 4 — Merge entradas + salidas → saldo_neto
#============================================================
df_evr = pd.merge(salidas, entradas, on=['provincia', 'anyo'], how='inner')
df_evr['saldo_neto'] = (df_evr['entradas'] - df_evr['salidas']).astype(int)
df_evr['entradas']  = df_evr['entradas'].astype(int)
df_evr['salidas']   = df_evr['salidas'].astype(int)
df_evr = df_evr.sort_values(['provincia', 'anyo']).reset_index(drop=True)

print(f"Shape: {df_evr.shape}")
print(f"\nNulos:\n{df_evr.isnull().sum()}")
print(f"\nEstadísticas:")
print(df_evr[['entradas','salidas','saldo_neto']].describe().round(0))
print(f"\nMadrid (entradas y salidas deben ser positivas y similares ~60-80k):")
print(df_evr[df_evr['provincia']=='Madrid'].to_string(index=False))

# Validación: total entradas == total salidas (flujo conservado)
print(f"\n✅ Total entradas: {df_evr['entradas'].sum():,}")
print(f"✅ Total salidas:  {df_evr['salidas'].sum():,}")
print(f"   Deben ser iguales (flujo conservado)")

Shape: (728, 5)

Nulos:
provincia     0
anyo          0
salidas       0
entradas      0
saldo_neto    0
dtype: int64

Estadísticas:
       entradas  salidas  saldo_neto
count     728.0    728.0       728.0
mean     9162.0   9162.0        -0.0
std     10928.0  10434.0     15102.0
min      1108.0   1332.0    -70485.0
25%      3788.0   4006.0     -4085.0
50%      6020.0   6195.0      -144.0
75%     10280.0  10502.0      3931.0
max     76886.0  79288.0     70589.0

Madrid (entradas y salidas deben ser positivas y similares ~60-80k):
provincia  anyo  salidas  entradas  saldo_neto
   Madrid  2008    79288     17936      -61352
   Madrid  2009    73490     10071      -63419
   Madrid  2010    72549      4125      -68424
   Madrid  2011    67771      3871      -63900
   Madrid  2012    62397      5822      -56575
   Madrid  2013    59207      3703      -55504
   Madrid  2014    57410      5705      -51705
   Madrid  2015    55077     13311      -41766
   Madrid  2016    55188      6379      -4

In [5]:
#============================================================
# Celda 5 — Guardar silver clean_evr_provincia_anio
#============================================================
out = df_evr[['provincia','anyo','entradas','salidas','saldo_neto']].copy()
out.to_parquet(SILVER / "clean_evr_provincia_anio.parquet", index=False)
out.to_csv(SILVER    / "clean_evr_provincia_anio.csv",     index=False)

print(f"✅ Guardado en {SILVER}")
print(f"   Shape: {out.shape}")
print(f"\nTop 5 por saldo_neto en 2021:")
print(out[out['anyo']==2021].sort_values('saldo_neto', ascending=False).head(5).to_string(index=False))
print(f"\nBottom 5 por saldo_neto en 2021:")
print(out[out['anyo']==2021].sort_values('saldo_neto').head(5).to_string(index=False))

✅ Guardado en output/evr/02-silver
   Shape: (728, 5)

Top 5 por saldo_neto en 2021:
        provincia  anyo  entradas  salidas  saldo_neto
        Tarragona  2021     62546    10628       51918
Valencia/València  2021     37083    17434       19649
         Palencia  2021     19526     1978       17548
            Soria  2021     17956     1481       16475
       Valladolid  2021     19236     5359       13877

Bottom 5 por saldo_neto en 2021:
     provincia  anyo  entradas  salidas  saldo_neto
        Madrid  2021      8115    78600      -70485
     Barcelona  2021     12047    50376      -38329
Balears, Illes  2021      3358    15509      -12151
        Málaga  2021      2619    14533      -11914
        Toledo  2021      5595    14297       -8702
